# Scraping for Urban Farms in Dallas County 

Things to do: 

In [4]:
import requests
import pandas as pd 
import geopandas as gpd 
from sklearn.cluster import DBSCAN
from rapidfuzz import fuzz # for dropping fuzzy duplicates 

The Dallas Coalition for Hunger Solutions hosts a [map](https://www.dallashunger.org/garden-locator-map) of urban farms, including community gardens, school food gardens, urban farms, market gardens, and teaching gardens. It is based on self-reporting through a form on the page, so it is likely not comprehensive (it doesn't even include Joppy Momma's Farm), but it's  an excellent starting point. 

In [2]:
# Scraping data from the map page
url = 'https://core.service.elfsight.com/p/boot/?page=https%3A%2F%2Fwww.dallashunger.org%2Fgarden-locator-map&w=5145b646-fd4c-4aa9-b6cf-d7e8b826e0ad'
headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)
raw_data = response.json()

# 1. Safely extract the widgets container
widgets_container = raw_data['data']['widgets']

# 2. Extract the first widget payload regardless of whether it is a list or a dict
if isinstance(widgets_container, dict):
    # If it's a dict, get the first value safely without using a key like 0
    first_widget_config = next(iter(widgets_container.values()))
elif isinstance(widgets_container, list):
    # If it's a list, look at the first element safely
    first_widget_config = widgets_container[0]
else:
    raise TypeError(f"Unexpected type for widgets: {type(widgets_container)}")

# 3. Drill down into the markers
markers = first_widget_config['data']['settings']['markers']

# 4. Check if markers is a list or a dictionary
if isinstance(markers, dict):
    # If markers is a dictionary, use from_dict orient='index'
    df = pd.DataFrame.from_dict(markers, orient='index')
elif isinstance(markers, list):
    # If markers is already a clean list of dictionaries, load it directly
    df = pd.DataFrame(markers)
else:
    raise TypeError(f"Unexpected type for markers: {type(markers)}")

print(f"Success! Extracted {len(df)} urban farms.")

# 5. Filter for just the essential spatial and descriptive data
relevant_cols = [col for col in ['title', 'address', 'lat', 'lng', 'categoryId'] if col in df.columns]

if relevant_cols:
    clean_df = df[relevant_cols]
    print("\nCleaned Data Sample:")
    print(clean_df.head())
    clean_df.to_csv("dallas_urban_farms.csv", index=False)
else:
    print("\nColumns found in the dataframe:")
    print(df.columns.tolist())
    print("\nFirst row sample:")
    print(df.iloc[0].to_dict())

Success! Extracted 191 urban farms.

Columns found in the dataframe:
['position', 'coordinates', 'icon', 'category', 'linkUrl', 'infoWindow', 'infoTitle', 'infoDescription', 'infoImage', 'infoAddress', 'infoSite', 'infoPhone', 'infoEmail', 'infoWorkingHours', 'infoWindowOpenedByDefault', 'markerClickAction', 'animation', 'id', 'chosen', 'selected']

First row sample:
{'position': '1513 S. Beltline Rd., Grand Prairie, TX 75052', 'coordinates': '32.7239892, -96.9960085', 'icon': None, 'category': 'afb9d952-8c58-405a-9ccc-cc944ec6ad6c', 'linkUrl': '', 'infoWindow': True, 'infoTitle': 'The Haven Community Garden', 'infoDescription': '<div>38 raised beds; 8 garden rows; 14 fruit trees; hours of operation: dawn till dusk; no annual fee; funded by Keep Grand Prairie Beautiful.</div>', 'infoImage': None, 'infoAddress': '', 'infoSite': '', 'infoPhone': '', 'infoEmail': ' sharon.dehnert@tx.rr.com', 'infoWorkingHours': '', 'infoWindowOpenedByDefault': False, 'markerClickAction': 'infoWindow', 'an

In [ ]:
# Saving to CSV 
clean_df.to_csv('dallasHungerDirectory.csv', index=False)

The second source was one I found on the City of Dallas's [Dallas Climate Action Website](https://www.dallasclimateaction.com/foodaccess). It hosts a GIS map that includes many urban farms and community gardens from 2022 and 2024. 

In [8]:
# Scraping the 2024 layer 

# 1. Set your endpoint (Make sure it ends in /query)
url = 'https://services2.arcgis.com/rwnOSbfKSwyTBcwN/arcgis/rest/services/DallasUrbanGarden2024_Geocoded/FeatureServer/1/query'

# 2. Setup the loop variables
all_features = []
offset = 0
chunk_size = 1000 # A safe number that won't trigger server limits
has_more_data = True

while has_more_data:
    # Define the query parameters
    params = {
        "where": "1=1",
        "outFields": "*",
        "returnGeometry": "true",
        "f": "geojson",
        "resultOffset": offset,
        "resultRecordCount": chunk_size
    }
    
    # Execute the request
    response = requests.get(url, params=params)
    response.raise_for_status() # Catch any HTTP errors immediately
    
    data = response.json()
    
    # Extract the chunk of data
    features = data.get("features", [])
    all_features.extend(features)
    
    # Check if we hit the end of the dataset
    # Esri often includes an "exceededTransferLimit" flag if there is more data waiting
    if data.get("exceededTransferLimit") or len(features) == chunk_size:
        offset += chunk_size
        print(f"Downloaded {len(all_features)} records so far...")
    else:
        has_more_data = False # We've reached the end

print(f"Extraction complete! Total records: {len(all_features)}")

# 3. Convert directly into a GeoDataFrame
# Because we requested standard GeoJSON, Geopandas can ingest the raw dictionary list directly
if all_features:
    # Reconstruct a valid GeoJSON object in memory
    geojson_payload = {
        "type": "FeatureCollection",
        "features": all_features
    }
    
    gdf = gpd.GeoDataFrame.from_features(geojson_payload["features"])
    
    # Set the default coordinate reference system to standard GPS (WGS84)
    gdf.set_crs(epsg=4326, inplace=True)
    
    print("\nSample of extracted data:")
    print(gdf.head())

Pinging the ArcGIS server...
Extraction complete! Total records: 59

Sample of extracted data:
                     geometry  ObjectID        Loc_name USER_City  \
0  POINT (-96.80498 32.74688)         1  CrmDallasStree    Dallas   
1  POINT (-96.86937 32.85665)         2  CrmDallasStree    Dallas   
2  POINT (-96.75866 32.77632)         3  CrmDallasStree    Dallas   
3   POINT (-96.7537 32.73491)         4  CrmDallasStree    Dallas   
4  POINT (-96.69398 32.84711)         5  CrmDallasStree    Dallas   

  USER_Subregion                                      USER_SiteName  \
0  Dallas County                            10 St. Community Garden   
1  Dallas County         Bachman Lake Together Family Center Garden   
2  Dallas County                                Big Tex Urban Farms   
3  Dallas County                                       Bonton Farms   
4  Dallas County  Central Lutheran Garden, at Central Lutheran C...   

   USER_OldId                                   USER_ARC_Addres

In [9]:
gdf.to_file("dallasClimateDirectory24", driver="GeoJSON")

In [3]:
# Scraping the 2022 layer

# 1. Set your endpoint (Make sure it ends in /query)
url = 'https://services2.arcgis.com/rwnOSbfKSwyTBcwN/arcgis/rest/services/UrbanGardensNov2022_Geocoded/FeatureServer/0/query'

# 2. Setup the loop variables
all_features = []
offset = 0
chunk_size = 1000 # A safe number that won't trigger server limits
has_more_data = True

while has_more_data:
    # Define the query parameters
    params = {
        "where": "1=1",
        "outFields": "*",
        "returnGeometry": "true",
        "f": "geojson",
        "resultOffset": offset,
        "resultRecordCount": chunk_size
    }
    
    # Execute the request
    response = requests.get(url, params=params)
    response.raise_for_status() # Catch any HTTP errors immediately
    
    data = response.json()
    
    # Extract the chunk of data
    features = data.get("features", [])
    all_features.extend(features)
    
    # Check if we hit the end of the dataset
    # Esri often includes an "exceededTransferLimit" flag if there is more data waiting
    if data.get("exceededTransferLimit") or len(features) == chunk_size:
        offset += chunk_size
        print(f"Downloaded {len(all_features)} records so far...")
    else:
        has_more_data = False # We've reached the end

print(f"Extraction complete! Total records: {len(all_features)}")

# 3. Convert directly into a GeoDataFrame
# Because we requested standard GeoJSON, Geopandas can ingest the raw dictionary list directly
if all_features:
    # Reconstruct a valid GeoJSON object in memory
    geojson_payload = {
        "type": "FeatureCollection",
        "features": all_features
    }
    
    gdf = gpd.GeoDataFrame.from_features(geojson_payload["features"])
    
    # Set the default coordinate reference system to standard GPS (WGS84)
    gdf.set_crs(epsg=4326, inplace=True)
    
    print("\nSample of extracted data:")
    print(gdf.head())

Pinging the ArcGIS server...
Extraction complete! Total records: 54

Sample of extracted data:
                     geometry  OBJECTID  Score  \
0  POINT (-96.76109 32.87116)         1  100.0   
1  POINT (-96.75054 32.86678)         2  100.0   
2   POINT (-96.7709 32.77334)         3  100.0   
3  POINT (-96.78581 32.78083)         4  100.0   
4  POINT (-96.73952 32.78156)         5  100.0   

                               Match_addr  \
0             8361 PARK LN, DALLAS, 75231   
1        6723 EASTRIDGE DR, DALLAS, 75231   
2         2641 JEFFRIES ST, DALLAS, 75215   
3  458 S GOOD LATIMER EXPY, DALLAS, 75201   
4          4815 SILVER AVE, DALLAS, 75223   

                                LongLabel               ShortLabel  \
0             8361 PARK LN, DALLAS, 75231             8361 PARK LN   
1        6723 EASTRIDGE DR, DALLAS, 75231        6723 EASTRIDGE DR   
2         2641 JEFFRIES ST, DALLAS, 75215         2641 JEFFRIES ST   
3  458 S GOOD LATIMER EXPY, DALLAS, 75201  458 S GOOD

In [11]:
gdf.to_file('dallasClimateDirectory22', driver='GeoJSON')

Now, we can put all three datasets together and deduplicate. 

In [35]:
# Reading in Dallas Climate Action Directory DFs
d24 = gpd.read_file('dallasClimateDirectory24')
d22 = gpd.read_file('dallasClimateDirectory22')

In [36]:
# Aligning CRS of 2022 and 2024 data
if d24.crs != d22.crs:
    print("CRS mismatch detected. Aligning projections...")
    d22 = d22.to_crs(d24.crs)

# Putting both 2024 and 2022 dataframes together 
climateDirect = pd.concat([d24, d22], ignore_index=True)

# Converting to GDF 
climateDirectGDF = gpd.GeoDataFrame(climateDirect, geometry='geometry', crs=d24.crs)

We need to filter out farms that do not generate revenue. We can select such farms based on category variables available in the datasets.

In [37]:
# checking which observations are missing the category variables of interest 
target_cols = ['USER_Wholesale', 'USER_Retail', 'USER_UrbanFarm', 'USER_MKTGRDN']
climateDirectGDF[climateDirectGDF[target_cols].isnull().any(axis=1)]

,ObjectID,Loc_name,USER_City,USER_Subregion,USER_SiteName,USER_OldId,USER_ARC_Address,Prod_Acres,USER_Existing,USER_Founded,...,X,Y,USER_OBJECTID,USER_ARC_Addres,USER_Street,USER_State,USER_len,USER_StreetName,USER_ZIP,USER_ProdAcres
57,58.0,CrmDallasStree,Dallas,Dallas County,West Dallas Multipurpose Center's Community Ga...,0.0,"2828 Fish Trap Road, Dallas, TX 75212",0.08,1.0,N/A,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
109,NaN,NaN,Dallas,None,Joppy Momma's Farm,NaN,NaN,NaN,NaN,None,...,2.506321e+06,6.948328e+06,54.0,None,"7724 Carbondale St, Dallas, TX 75216",Texas,36.0,7724 Carbondale St,75216.0,NaN


These two observations don't need to be included because one of them does not sell its produce and the other (Joppy Momma's) has another observation. 

In [38]:
# Filtering to only those that fall into one of these categories
climateDirectGDF = climateDirectGDF[(climateDirectGDF[target_cols] == 1).any(axis=1)]

In [39]:
# Dropping observations with duplicate geometries
climateDirectGDF = climateDirectGDF.drop_duplicates(subset=['geometry'], ignore_index=True)

Next, we can execute a similar process with the Dallas Hunger Solutions Data

In [40]:
# Loading as a DF 
dHunger = pd.read_csv('dallasHungerDirectory.csv')

In [41]:
# Filtering by category to revenue-generating operations
dHunger = dHunger[dHunger['category'].isin
    (['5d986013-f56f-416f-bd10-3f083e69e302', '97d6907d-b8ba-473d-9c4a-a4834c9da7f0'])] # codes for "urban farm" and "market garden" 

In [42]:
# Converting unified 'coordinates' varaible to X and Y coordinates to convert to GDF
split_coords = dHunger['coordinates'].str.split(',', expand=True)

dHungerGDF = gpd.GeoDataFrame(
    dHunger,
    geometry=gpd.points_from_xy(split_coords[1].astype(float), split_coords[0].astype(float)),
    crs="EPSG:4326"
)

In [43]:
# Aligning CRS of unified Dallas Climate Action and Hunger Solutions DFs 
if climateDirectGDF.crs != dHungerGDF.crs:
    print("CRS mismatch detected. Aligning projections...")
    climateDirectGDF = climateDirectGDF.to_crs(dHungerGDF.crs)

# Putting both DFs together 
dallasFarmsDF = pd.concat([climateDirectGDF, dHungerGDF], ignore_index=True)

# Converting to GDF
dallasFarmsGDF = gpd.GeoDataFrame(dallasFarmsDF, geometry='geometry', crs=dHungerGDF.crs)

In [44]:
# Dropping simple duplicates based on geometry again 
dallasFarmsGDF = dallasFarmsGDF.drop_duplicates(subset=['geometry'], ignore_index=True)

In [45]:
# Filtering only to relevant variables: name, address, and geometry 
dallasFarmsGDF = dallasFarmsGDF[['USER_SiteName', 'USER_ARC_Address', 'geometry', 'position', 'infoTitle']]

In [46]:
# Find rows where both name columns are not null (there should be none)
conflicts = dallasFarmsGDF[dallasFarmsGDF['USER_SiteName'].notna() & dallasFarmsGDF['infoTitle'].notna()]

print(f"Number of conflicting rows: {len(conflicts)}")

Number of conflicting rows: 0


In [47]:
# Find rows where both address columns are not null (there should be none)
conflicts2 = dallasFarmsGDF[dallasFarmsGDF['USER_SiteName'].notna() & dallasFarmsGDF['infoTitle'].notna()]

print(f"Number of conflicting rows: {len(conflicts2)}")

Number of conflicting rows: 0


In [48]:
# Putting the different name and address columns together 
dallasFarmsGDF['SiteName'] = dallasFarmsGDF['USER_SiteName'].combine_first(dallasFarmsGDF['infoTitle'])
dallasFarmsGDF['Address'] = dallasFarmsGDF['USER_ARC_Address'].combine_first(dallasFarmsGDF['position'])

# Removing extraneous columns
dallasFarmsGDF = dallasFarmsGDF[['SiteName', 'Address', 'geometry']]

In [15]:
dallasFarmsGDF.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 35 entries, 0 to 34
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   SiteName  35 non-null     object  
 1   Address   30 non-null     object  
 2   geometry  35 non-null     geometry
dtypes: geometry(1), object(2)
memory usage: 972.0+ bytes


In [13]:
dallasFarmsGDF.sort_values(by='SiteName')

,SiteName,Address,geometry
0,10 St. Community Garden,"1300 East Clarendon Dr., Dallas, TX, 75203",POINT (-96.80498 32.74688)
113,10 St. Community Garden,"1300 E. Clarendon Dr., Dallas, TX, 75203",POINT (-96.80502 32.74701)
94,10 St. Community Garden,NaN,POINT (-96.8053 32.74687)
114,A Special Blend,"2378 E. Ledbetter Dr., Dallas, TX, 75216",POINT (-96.78242 32.68898)
115,Adamson High School Garden,"309 E 9th St., Dallas, TX, 75203",POINT (-96.82019 32.74824)
...,...,...,...
58,Worth Street Community Garden,"513 N Peak St, Dallas, Texas, 75246",POINT (-96.77293 32.79315)
262,Worth Street Community Garden,"517 N. Peak St., Dallas, TX, 75046",POINT (-96.77315 32.79313)
83,Worth Street Garden,NaN,POINT (-96.77258 32.79299)
273,Wyatt-Shockley Community Garden,"3610 Dunbar Street, Dallas, Texas 76215",POINT (-96.75996 32.76743)


In [49]:
# Using both DBSCAN and fuzz to get rid of duplicates
# 1. Project to meters for accurate distance measuring
gdf_meters = dallasFarmsGDF.to_crs(epsg=3857)

# 2. Run DBSCAN with a massive search radius (e.g., 500 meters)
# We can make this huge because DBSCAN isn't deleting anything anymore; 
# it is just grouping points for the text matcher to look at.
coords = gdf_meters.get_coordinates()
clustering = DBSCAN(eps=500, min_samples=1).fit(coords)

# Assign the cluster IDs back to the original dataframe
dallasFarmsGDF['cluster_id'] = clustering.labels_

# 3. Prepare the text column (crucial for exact matches)
dallasFarmsGDF['clean_name'] = dallasFarmsGDF['SiteName'].astype(str).str.lower()
dallasFarmsGDF['clean_name'] = dallasFarmsGDF['clean_name'].str.replace(r'[^\w\s]', '', regex=True)

indices_to_drop = set()

# 4. Loop through the organic DBSCAN clusters instead of the rigid grid
for _, group in dallasFarmsGDF.groupby('cluster_id'):
    
    # Skip clusters that only have 1 point
    if len(group) < 2:
        continue
        
    records = list(group['clean_name'].dropna().items())
    
    # Run the fuzzy matcher inside the organic cluster
    for i in range(len(records)):
        idx1, name1 = records[i]
        if idx1 in indices_to_drop:
            continue 

        for j in range(i + 1, len(records)):
            idx2, name2 = records[j]
            if idx2 in indices_to_drop:
                continue

            score = fuzz.token_set_ratio(str(name1), str(name2))
            
            if score >= 85:
                indices_to_drop.add(idx2)

# 5. Drop the duplicates and clean up
dallasFarmsCleanGDF = dallasFarmsGDF.drop(index=list(indices_to_drop))
dallasFarmsCleanGDF = dallasFarmsCleanGDF.drop(columns=['cluster_id', 'clean_name'])

In [19]:
dallasFarmsCleanGDF.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 25 entries, 0 to 32
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   SiteName  25 non-null     object  
 1   Address   23 non-null     object  
 2   geometry  25 non-null     geometry
dtypes: geometry(1), object(2)
memory usage: 800.0+ bytes


In [20]:
# Checking for persistent duplicates
dallasFarmsCleanGDF['SiteName'].value_counts()

SiteName
Elmwood Farm                                            2
Big Tex Urban Farms                                     1
Dallas Half Acre Farm                                   1
Bonton Farms                                            1
Disciple City Church Orchard                            1
Grozilla                                                1
Hatcher Station Training Farm at Restorative Farm       1
ILC Aquaponics                                          1
Joppy Momma's Farm                                      1
Liberty Street Garden, at The Meadows Foundation        1
Mama Ida's Teaching Garden, at Dallas Farmers Market    1
Moss Haven Farms, at Moss Haven Elementary School       1
Old East Dallas Urban Garden                            1
Owenwood Farm & Community Garden                        1
The Seedling Farm at MLK Freedom Garden                 1
WE over Me Farm, at Paul Quinn College                  1
East Dallas Community Garden                            1
Sunny

One of the Elmwood Farms is a completely different location, but there is only one listed on Google Maps. We will take the address for that location and drop the other one.

In [50]:
# Dropping the last duplicate 
dallasFarmsCleanGDF[dallasFarmsCleanGDF['SiteName'] == 'Elmwood Farm']
dallasFarmsCleanGDF = dallasFarmsCleanGDF.drop(32)

In [22]:
# Saving to clean GeoJSON 
dallasFarmsCleanGDF.to_file('dallasFarmsClean.geojson', driver='GeoJSON')

There is also a dataset hosted on the City of Dallas's [GIS Portal](https://egisdata-dallasgis.hub.arcgis.com/datasets/DallasGIS::dallasurbangarden2024-geocoded/explore?location=32.816194%2C-96.764276%2C10) that purportedly hosts the same data on urban agriculture. We can download and compare our results. 

In [23]:
# Loading the GIS Portal dataset
dallasGISHub = gpd.read_file('DallasUrbanGarden.geojson')
dallasGISHub.columns

Index(['ObjectID', 'Loc_name', 'USER_City', 'USER_Subregion', 'USER_SiteName',
       'USER_OldId', 'USER_ARC_Address', 'Prod_Acres', 'USER_Existing',
       'USER_Founded', 'USER_ContactNam', 'USER_Phone_1', 'USER_Email',
       'USER_Website', 'USER_Descriptio', 'USER_Wholesale', 'USER_Retail',
       'USER_UrbanFarm', 'USER_ComGRDN', 'USER_HomeGRDN', 'USER_MKTGRDN',
       'USER_SchoolGRDN', 'USER_TeachingGR', 'USER_FaithGRDN', 'USER_ParkGRDN',
       'USER_PantryGRDN', 'USER_OtherGRDN', 'USER_IndoorProd',
       'USER_NonSoilPro', 'USER_OtherProdu', 'USER_Public_', 'USER_Private',
       'USER_Nonprofit', 'USER_Programs', 'USER_Partners', 'USER_Management',
       'USER_Crops', 'USER_Storage', 'USER_Distributi', 'USER_Composting',
       'USER_Sales', 'USER_ADA', 'USER_SNAP', 'USER_Zoning', 'USER_Funding',
       'USER_LandTenure', 'USER_PopServed', 'USER_Needs', 'USER_Notes',
       'USER_SiteCount', 'USER_Source', 'USER_DataUpdate', 'USER_ParcelType',
       'USER_Risks', 'USER_O

The columns seem to be exactly the same as in the Climate Solutions dataset. 

In [24]:
# Filtering out non-revenue-generating farms
dallasGISHub = dallasGISHub[(dallasGISHub[target_cols] == 1).any(axis=1)]

# Dropping exact duplicates absed on geometry
dallasGISHub = dallasGISHub.drop_duplicates(subset=['geometry'], ignore_index=True)

# Filtering only to relevant columns
dallasGISHub  = dallasGISHub[['USER_SiteName', 'USER_ARC_Address', 'geometry']]

In [25]:
# Deduplicating with DBSCAN and fuzz
# 1. Project to meters for accurate distance measuring
gdf_meters = dallasGISHub.to_crs(epsg=3857)

# 2. Run DBSCAN with a massive search radius (e.g., 500 meters)
# We can make this huge because DBSCAN isn't deleting anything anymore; 
# it is just grouping points for the text matcher to look at.
coords = gdf_meters.get_coordinates()
clustering = DBSCAN(eps=500, min_samples=1).fit(coords)

# Assign the cluster IDs back to the original dataframe
dallasGISHub['cluster_id'] = clustering.labels_

# 3. Prepare the text column (crucial for exact matches)
dallasGISHub['clean_name'] = dallasGISHub['USER_SiteName'].astype(str).str.lower()
dallasGISHub['clean_name'] = dallasGISHub['clean_name'].str.replace(r'[^\w\s]', '', regex=True)

indices_to_drop = set()

# 4. Loop through the organic DBSCAN clusters instead of the rigid grid
for _, group in dallasGISHub.groupby('cluster_id'):
    
    # Skip clusters that only have 1 point
    if len(group) < 2:
        continue
        
    records = list(group['clean_name'].dropna().items())
    
    # Run the fuzzy matcher inside the organic cluster
    for i in range(len(records)):
        idx1, name1 = records[i]
        if idx1 in indices_to_drop:
            continue 

        for j in range(i + 1, len(records)):
            idx2, name2 = records[j]
            if idx2 in indices_to_drop:
                continue

            score = fuzz.token_set_ratio(str(name1), str(name2))
            
            if score >= 85:
                indices_to_drop.add(idx2)

# 5. Drop the duplicates and clean up
dallasGISHubClean = dallasGISHub.drop(index=list(indices_to_drop))
dallasGISHubClean = dallasGISHubClean.drop(columns=['cluster_id', 'clean_name'])

In [28]:
dallasGISHubClean.head(16)

,USER_SiteName,USER_ARC_Address,geometry
0,Big Tex Urban Farms,3921 Martin Luther King Jr Blvd Dallas Texas 7...,POINT (-96.75866 32.77632)
1,Bonton Farms,"6915 Bexar St, Dallas, TX 75215",POINT (-96.7537 32.73491)
2,Dallas Half Acre Farm,"13608 Fernheath Ln, Dallas, TX 75253",POINT (-96.58058 32.69072)
3,Disciple City Church Orchard,"1304 S Hampton Rd, Dallas, TX 75208",POINT (-96.85675 32.73174)
4,Elmwood Farm,"1014 Nolte Dr, Dallas, TX 75208",POINT (-96.83991 32.73385)
5,Grozilla,3921 Martin Luther King Jr Blvd Dallas TX 75210,POINT (-96.75888 32.77659)
6,Hatcher Station Training Farm at Restorative Farm,"4527 Scyene Rd, Dallas, 75210",POINT (-96.74432 32.76611)
7,ILC Aquaponics,"7500 W Camp Wisdom Rd, Dallas, TX 75236",POINT (-96.95406 32.66399)
8,Joppy Momma's Farm,"7724 Carbondale St, Dallas, TX 75216",POINT (-96.75087 32.71668)
9,"Liberty Street Garden, at The Meadows Foundation","510 Liberty St, Dallas, TX 75204",POINT (-96.78564 32.78902)


All of the entries are found in the dataset I put together. I just ended up with slightly more sites. I'm going to opt to use my dataset instead. 

In [26]:
dallasGISHubClean.to_file('dallasGISPortal.geojson', driver='GeoJSON') 

With this clean list, I manually verified that each of the sites in our fit our criteria of: 
1. Being located within Dallas County
2. Revenue-generating
   
Because the data were collected in past years, I also checked that none of the sites had moved locations, changed names, or no longer exist. After this final audit, I ended up with the following farms. 

In [52]:
final_list = [1, 2, 4, 5, 6, 8, 9, 13, 15, 16, 21]
dallasFarmsCleanGDF = dallasFarmsCleanGDF[dallasFarmsCleanGDF.index.isin(final_list)]
dallasFarmsCleanGDF.to_file('dallasFarmsFinal.geojson', driver='GeoJSON')